In [ ]:
import sys
sys.path.append("PyDI/PyDI")
from cleaners.rag_cleaner import RAGCleaner
import pandas as pd
import ollama
import os

In [ ]:
# Knowledge base = datasets 1, 3, 4 combined
kb = pd.concat([
    pd.read_json("normalized_products/dataset_2_normalized.json"),
    pd.read_json("normalized_products/dataset_3_normalized.json"),
    pd.read_json("normalized_products/dataset_4_normalized.json")
], ignore_index=True)

# Dataset to clean
df = pd.read_json("normalized_products/dataset_1_normalized.json")

In [ ]:
kb.head()

In [ ]:
class OllamaLLMWrapper:
    def __init__(self, model="llama3"):
        self.model = model

    def generate(self, prompt):
        response = ollama.chat(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a structured data repair tool. "
                        "You ONLY output tokens in the format VALUE:<value>. "
                        "You NEVER write sentences, explanations, or any other text. "
                        "Your entire response must be a single line: VALUE:<value>"
                    )
                },
                {"role": "user", "content": prompt}
            ]
        )
        return response["message"]["content"].strip()

In [ ]:
rag_cleaner = RAGCleaner(
    knowledge_base=kb,
    target_attributes=[
        "model_number",
        "storage_size",
        "bus_type",
        "interface_type",
        "form_factor",
        "vram",
        "storage_connection_type"
    ],
    llm=OllamaLLMWrapper("llama3"),
    top_k=2
)

In [ ]:
cols_to_track = [
    "model_number", "storage_size", "bus_type",
    "interface_type", "form_factor", "vram", "storage_connection_type"
]

In [ ]:
null_before = df[cols_to_track].isnull().sum()
print("=== NULL COUNTS BEFORE CLEANING ===")
print(null_before)
print(f"Total: {null_before.sum()}\n")

In [ ]:
df_test = df.head(2)

In [ ]:
# cleaned_df = rag_cleaner.clean(df_test)
cleaned_df = rag_cleaner.clean(df)

In [ ]:
# print(cleaned_df[[
#     "title",
#     "model_number",
#     "storage_size",
#     "bus_type",
#     "interface_type",
#     "form_factor",
#     "vram",
#     "storage_connection_type"
# ]])

In [ ]:
# After cleaning
null_after = cleaned_df[cols_to_track].isnull().sum()
print("\n=== NULL COUNTS AFTER CLEANING ===")
print(null_after)
print(f"Total: {null_after.sum()}\n")

In [ ]:
# Summary
filled = null_before.sum() - null_after.sum()
total = null_before.sum()
print(f"=== SUMMARY ===")
print(f"Filled: {filled} / {total} ({100*filled/total:.1f}% fill rate)")